# Outcome Predictor — Model 2 (MDP Architecture)

**What this notebook does:**
Given a play type (run/pass) and game state, predict:
1. **Yards gained** — regression head (continuous)
2. **Turnover flag** — binary classification (interception or fumble lost)
3. **Receiver position group** (pass plays only) — WR / TE / RB — which position group was targeted
4. **Touchdown flag** — binary classification

These four outputs let `simulate_game.py` build a causally consistent play log:  
yards determine field position, turnovers flip possession, receiver position determines which player to credit, TDs update the score.

**Architecture:** One shared encoder → four output heads (multi-task learning).  
Run plays and pass plays are trained together with a `play_type` embedding.

**Before running:** Switch runtime to T4 GPU — Runtime → Change runtime type → T4 GPU

**Output artifacts (place in `backend/python_backend/outcome_model_weights/`):**
- `outcome_model.pt` — PyTorch weights
- `outcome_feature_meta.json` — feature names, cardinalities, bucket definitions
- `outcome_config.json` — architecture config for inference service
- `outcome_yards_scaler.pkl` — StandardScaler for yards (regression target)

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('WARNING: No GPU. Switch to T4 GPU runtime before proceeding.')

## 1: Install dependencies

In [ ]:
%%capture
!pip install nflreadpy scikit-learn pandas numpy torch --quiet

## 2: Load play-by-play data

In [ ]:
import nflreadpy as nfl
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

SEASONS = list(range(2015, 2026))
print(f'Loading PBP for seasons: {SEASONS}')
pbp_raw = nfl.load_pbp(SEASONS).to_pandas()
print(f'Loaded: {pbp_raw.shape[0]:,} plays x {pbp_raw.shape[1]} columns')

## 3: Filter to run and pass plays only

Outcome model only covers scrimmage plays where yards/turnovers/TDs apply.  
Punt and FG outcomes are handled by fixed distributions in the sim engine.

In [ ]:
df = pbp_raw[
    pbp_raw['play_type'].isin({'run', 'pass'}) &
    pbp_raw['down'].notna() &
    pbp_raw['ydstogo'].notna() &
    pbp_raw['yardline_100'].notna() &
    pbp_raw['score_differential'].notna() &
    pbp_raw['qtr'].notna() &
    pbp_raw['yards_gained'].notna()
].copy()

# Fill optional columns
df['shotgun']      = df['shotgun'].fillna(0).astype(int)
df['goal_to_go']   = df['goal_to_go'].fillna(0).astype(int)
df['air_yards']    = df['air_yards'].fillna(0)
df['pass_length']  = df['pass_length'].fillna('unknown')
df['receiver_player_name'] = df['receiver_player_name'].fillna('')

print(f'After filtering: {len(df):,} plays')
print(f'  Run plays:  {(df["play_type"]=="run").sum():,}')
print(f'  Pass plays: {(df["play_type"]=="pass").sum():,}')
print(f'\nYards gained range: {df["yards_gained"].min():.0f} to {df["yards_gained"].max():.0f}')
print(f'Median yards gained: {df["yards_gained"].median():.1f}')

## 4: Build target labels

Four prediction targets:
- `target_yards` — yards gained (clipped to [-10, 50] to reduce outlier influence)
- `target_turnover` — 1 if interception or fumble lost on this play
- `target_td` — 1 if touchdown scored on this play
- `target_receiver_pos` — for pass plays: 0=WR, 1=TE, 2=RB (inferred from position columns)

In [ ]:
# Yards: clip extreme outliers
df['target_yards'] = df['yards_gained'].clip(-10, 50).astype(float)

# Turnover: interception OR fumble lost
df['target_turnover'] = (
    (df.get('interception', pd.Series(0, index=df.index)).fillna(0) == 1) |
    (df.get('fumble_lost', pd.Series(0, index=df.index)).fillna(0) == 1)
).astype(int)

# TD: any touchdown on this play
df['target_td'] = df.get('touchdown', pd.Series(0, index=df.index)).fillna(0).astype(int)

# Receiver position group (pass plays only)
# nflverse has receiver_player_name but not always receiver position directly
# Use pass_location + air_yards as proxies, and check available position columns
pos_col = None
for col in ['receiver_player_position', 'receiver_position', 'position']:
    if col in df.columns:
        pos_col = col
        break

def infer_receiver_pos_group(row):
    """0=WR, 1=TE, 2=RB"""
    if row['play_type'] != 'pass':
        return -1  # not applicable for run plays
    if pos_col and pd.notna(row.get(pos_col, None)):
        pos = str(row[pos_col]).upper()
        if pos in ('WR', 'DB', 'CB'):  return 0
        if pos in ('TE',):              return 1
        if pos in ('RB', 'HB', 'FB'):  return 2
    # Fallback: infer from air yards
    # Short passes behind LOS more likely RB/TE, deep passes more likely WR
    air = row.get('air_yards', 0) or 0
    if air < 0:   return 2   # behind LOS = screen = RB
    elif air < 6: return 1   # short = TE
    else:         return 0   # medium/deep = WR

print('Building receiver position labels (this takes ~30s on 1M rows)...')
df['target_receiver_pos'] = df.apply(infer_receiver_pos_group, axis=1)

print('\nTarget distributions:')
print(f'  Turnovers: {df["target_turnover"].mean()*100:.2f}% of plays')
print(f'  Touchdowns: {df["target_td"].mean()*100:.2f}% of plays')
print(f'  Receiver pos (pass only):')
pass_df = df[df['play_type'] == 'pass']
for code, label in [(0,'WR'),(1,'TE'),(2,'RB')]:
    pct = (pass_df['target_receiver_pos'] == code).mean() * 100
    print(f'    {label}: {pct:.1f}%')

## 5: Feature engineering

Same game-state buckets as Model 1, plus play_type embedding and air_yards bucket for pass plays.

In [ ]:
def dist_bucket(x):
    if x <= 2:    return 0
    elif x <= 6:  return 1
    elif x <= 10: return 2
    else:         return 3

def yard_zone(x):
    if x >= 80:   return 0
    elif x >= 60: return 1
    elif x >= 40: return 2
    elif x >= 20: return 3
    elif x >= 5:  return 4
    else:         return 5

def score_bucket(x):
    if x <= -17:  return 0
    elif x <= -7: return 1
    elif x <= 6:  return 2
    elif x <= 16: return 3
    else:         return 4

def air_yards_bucket(x):
    """Pass depth: screen/behind, short, medium, deep"""
    if x < 0:     return 0  # screen / behind LOS
    elif x <= 5:  return 1  # short
    elif x <= 15: return 2  # medium
    else:         return 3  # deep

df['feat_play_type']  = (df['play_type'] == 'pass').astype(int)   # 0=run, 1=pass
df['feat_down']       = df['down'].astype(int) - 1
df['feat_dist']       = df['ydstogo'].apply(dist_bucket)
df['feat_zone']       = df['yardline_100'].apply(yard_zone)
df['feat_score']      = df['score_differential'].apply(score_bucket)
df['feat_qtr']        = (df['qtr'].clip(1, 5) - 1).astype(int)
df['feat_shotgun']    = df['shotgun']
df['feat_goal_to_go'] = df['goal_to_go']
df['feat_air_yards']  = df['air_yards'].apply(air_yards_bucket)   # 0 for run plays

FEATURE_COLS = [
    'feat_play_type', 'feat_down', 'feat_dist', 'feat_zone',
    'feat_score', 'feat_qtr', 'feat_shotgun', 'feat_goal_to_go',
    'feat_air_yards',
]

FEAT_CARDINALITY = {
    'feat_play_type':  2,
    'feat_down':       4,
    'feat_dist':       4,
    'feat_zone':       6,
    'feat_score':      5,
    'feat_qtr':        5,
    'feat_shotgun':    2,
    'feat_goal_to_go': 2,
    'feat_air_yards':  4,
}

print('Feature engineering done.')
print(df[FEATURE_COLS].head(3))

## 6: Scale yards target + train/val split

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import pickle, json, os

X = df[FEATURE_COLS].values.astype(np.int64)
y_yards    = df['target_yards'].values.astype(np.float32)
y_turnover = df['target_turnover'].values.astype(np.int64)
y_td       = df['target_td'].values.astype(np.int64)
y_rec_pos  = df['target_receiver_pos'].values.astype(np.int64)

# Scale yards to zero mean, unit variance (regression trains much better this way)
yards_scaler = StandardScaler()
y_yards_scaled = yards_scaler.fit_transform(y_yards.reshape(-1, 1)).flatten().astype(np.float32)
print(f'Yards: mean={yards_scaler.mean_[0]:.2f}, std={yards_scaler.scale_[0]:.2f}')

idx = np.arange(len(X))
idx_train, idx_val = train_test_split(idx, test_size=0.10, random_state=42)

X_train, X_val         = X[idx_train], X[idx_val]
yw_train, yw_val       = y_yards_scaled[idx_train], y_yards_scaled[idx_val]
yt_train, yt_val       = y_turnover[idx_train], y_turnover[idx_val]
ytd_train, ytd_val     = y_td[idx_train], y_td[idx_val]
yrp_train, yrp_val     = y_rec_pos[idx_train], y_rec_pos[idx_val]

print(f'Train: {len(X_train):,}  Val: {len(X_val):,}')

## 7: Define the multi-task model

Shared embedding encoder → 4 heads:
- **yards_head**: Linear → scalar (regression)
- **turnover_head**: Linear → 2 logits (binary CE)
- **td_head**: Linear → 2 logits (binary CE)
- **receiver_pos_head**: Linear → 3 logits (3-class CE, only meaningful for pass plays)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

EMB_DIM = 8
HIDDEN  = 512
DROPOUT = 0.3


class OutcomeMLP(nn.Module):
    def __init__(self, feat_cardinality: dict, emb_dim: int, hidden: int):
        super().__init__()
        self.feat_names = list(feat_cardinality.keys())
        self.embeddings = nn.ModuleList([
            nn.Embedding(card, emb_dim) for card in feat_cardinality.values()
        ])
        in_dim = len(feat_cardinality) * emb_dim

        self.encoder = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden),
            nn.LayerNorm(hidden),
            nn.SiLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, hidden // 2),
            nn.LayerNorm(hidden // 2),
            nn.SiLU(),
        )

        head_in = hidden // 2
        self.yards_head       = nn.Linear(head_in, 1)
        self.turnover_head    = nn.Linear(head_in, 2)
        self.td_head          = nn.Linear(head_in, 2)
        self.receiver_pos_head = nn.Linear(head_in, 3)

    def forward(self, x: torch.Tensor):
        embs = [self.embeddings[i](x[:, i]) for i in range(len(self.feat_names))]
        h = self.encoder(torch.cat(embs, dim=-1))
        return (
            self.yards_head(h).squeeze(-1),       # (B,) continuous
            self.turnover_head(h),                 # (B, 2) logits
            self.td_head(h),                       # (B, 2) logits
            self.receiver_pos_head(h),             # (B, 3) logits
        )

    def predict(self, x: torch.Tensor, yards_scaler):
        """Returns (yards_float, turnover_prob, td_prob, receiver_pos_probs)"""
        self.eval()
        with torch.no_grad():
            y_hat, to_logits, td_logits, rp_logits = self.forward(x)
        yards_scaled = y_hat.cpu().numpy()
        yards = yards_scaler.inverse_transform(yards_scaled.reshape(-1, 1)).flatten()
        to_prob  = torch.softmax(to_logits, dim=-1)[:, 1].cpu().numpy()
        td_prob  = torch.softmax(td_logits, dim=-1)[:, 1].cpu().numpy()
        rp_probs = torch.softmax(rp_logits, dim=-1).cpu().numpy()
        return yards, to_prob, td_prob, rp_probs


model = OutcomeMLP(FEAT_CARDINALITY, EMB_DIM, HIDDEN).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Parameters: {n_params:,}')

## 8: Train

Multi-task loss = yards_MSE + turnover_CE + td_CE + receiver_pos_CE (pass plays only).  
Each head is weighted to keep losses on the same scale.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

EPOCHS     = 60
BATCH_SIZE = 4096
LR         = 3e-3

# Loss weights (tune if one head dominates)
W_YARDS    = 1.0
W_TURNOVER = 2.0   # upweight: rare but critical
W_TD       = 2.0   # upweight: rare but critical
W_REC_POS  = 0.5   # softer: position group is a proxy, not ground truth

X_tr  = torch.LongTensor(X_train).to(DEVICE)
yw_tr = torch.FloatTensor(yw_train).to(DEVICE)
yt_tr = torch.LongTensor(yt_train).to(DEVICE)
ytd_tr = torch.LongTensor(ytd_train).to(DEVICE)
yrp_tr = torch.LongTensor(yrp_train).to(DEVICE)

X_vl  = torch.LongTensor(X_val).to(DEVICE)
yw_vl = torch.FloatTensor(yw_val).to(DEVICE)
yt_vl = torch.LongTensor(yt_val).to(DEVICE)
ytd_vl = torch.LongTensor(ytd_val).to(DEVICE)
yrp_vl = torch.LongTensor(yrp_val).to(DEVICE)

# Mask: receiver pos head only trains on pass plays
pass_mask_tr = (X_tr[:, 0] == 1)  # feat_play_type == 1 (pass)
pass_mask_vl = (X_vl[:, 0] == 1)

train_ds = TensorDataset(X_tr, yw_tr, yt_tr, ytd_tr, yrp_tr)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=LR, steps_per_epoch=len(train_dl), epochs=EPOCHS
)

best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, ywb, ytb, ytdb, yrpb in train_dl:
        optimizer.zero_grad()
        y_hat, to_logits, td_logits, rp_logits = model(xb)

        loss_yards = F.mse_loss(y_hat, ywb)
        loss_to    = F.cross_entropy(to_logits, ytb)
        loss_td    = F.cross_entropy(td_logits, ytdb)

        # Receiver pos: only on pass plays in this batch
        pass_mask_b = (xb[:, 0] == 1)
        if pass_mask_b.sum() > 0:
            loss_rp = F.cross_entropy(rp_logits[pass_mask_b], yrpb[pass_mask_b])
        else:
            loss_rp = torch.tensor(0.0, device=DEVICE)

        loss = W_YARDS * loss_yards + W_TURNOVER * loss_to + W_TD * loss_td + W_REC_POS * loss_rp
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()

    if epoch % 10 == 0 or epoch == 1:
        model.eval()
        with torch.no_grad():
            y_hat_v, to_v, td_v, rp_v = model(X_vl)
            vl_yards = F.mse_loss(y_hat_v, yw_vl).item()
            vl_to    = F.cross_entropy(to_v, yt_vl).item()
            vl_td    = F.cross_entropy(td_v, ytd_vl).item()
            vl_rp    = F.cross_entropy(rp_v[pass_mask_vl], yrp_vl[pass_mask_vl]).item() if pass_mask_vl.sum() > 0 else 0
            vl_total = W_YARDS*vl_yards + W_TURNOVER*vl_to + W_TD*vl_td + W_REC_POS*vl_rp

            to_acc  = (to_v.argmax(-1) == yt_vl).float().mean().item()
            td_acc  = (td_v.argmax(-1) == ytd_vl).float().mean().item()

        print(f'Epoch {epoch:3d} | train {total_loss/len(train_dl):.4f} '
              f'| val_total {vl_total:.4f} | yards_mse {vl_yards:.4f} '
              f'| to_acc {to_acc*100:.1f}% | td_acc {td_acc*100:.1f}%')

        if vl_total < best_val_loss:
            best_val_loss = vl_total
            torch.save(model.state_dict(), '/tmp/outcome_model_best.pt')

print(f'\nBest val loss: {best_val_loss:.4f}')

## 9: Validation — yards distribution + calibration check

In [ ]:
model.load_state_dict(torch.load('/tmp/outcome_model_best.pt', map_location=DEVICE))
model.eval()

with torch.no_grad():
    y_hat_v, to_v, td_v, rp_v = model(X_vl)

# Yards: un-scale and compare
pred_yards = yards_scaler.inverse_transform(y_hat_v.cpu().numpy().reshape(-1,1)).flatten()
true_yards = y_yards[idx_val]

print('=== Yards prediction ===')
print(f'True  — mean: {true_yards.mean():.2f}  median: {np.median(true_yards):.2f}  std: {true_yards.std():.2f}')
print(f'Pred  — mean: {pred_yards.mean():.2f}  median: {np.median(pred_yards):.2f}  std: {pred_yards.std():.2f}')

# Split by play type
pt_val = X_val[:, 0]  # feat_play_type
for pt, label in [(0,'Run'),(1,'Pass')]:
    mask = pt_val == pt
    print(f'\n  {label} plays:')
    print(f'    True  mean={true_yards[mask].mean():.2f}  median={np.median(true_yards[mask]):.2f}')
    print(f'    Pred  mean={pred_yards[mask].mean():.2f}  median={np.median(pred_yards[mask]):.2f}')

print('\n=== Turnover calibration ===')
to_probs = torch.softmax(to_v, dim=-1)[:, 1].cpu().numpy()
print(f'Mean predicted turnover prob: {to_probs.mean()*100:.2f}%')
print(f'True turnover rate:           {y_turnover[idx_val].mean()*100:.2f}%')

print('\n=== TD calibration ===')
td_probs = torch.softmax(td_v, dim=-1)[:, 1].cpu().numpy()
print(f'Mean predicted TD prob: {td_probs.mean()*100:.2f}%')
print(f'True TD rate:           {y_td[idx_val].mean()*100:.2f}%')

print('\n=== Receiver pos calibration (pass plays) ===')
pass_mask_v_np = pt_val == 1
rp_probs = torch.softmax(rp_v[torch.BoolTensor(pass_mask_v_np).to(DEVICE)], dim=-1).cpu().numpy()
for i, label in enumerate(['WR','TE','RB']):
    pred_pct = rp_probs[:, i].mean() * 100
    true_pct = (y_rec_pos[idx_val][pass_mask_v_np] == i).mean() * 100
    print(f'  {label}: pred={pred_pct:.1f}%  true={true_pct:.1f}%')

## 10: Save all artifacts

In [ ]:
import shutil

SAVE_DIR = '/tmp/outcome_model_weights'
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Model weights
shutil.copy('/tmp/outcome_model_best.pt', f'{SAVE_DIR}/outcome_model.pt')
print('Saved outcome_model.pt')

# 2. Yards scaler
with open(f'{SAVE_DIR}/outcome_yards_scaler.pkl', 'wb') as f:
    pickle.dump(yards_scaler, f)
print('Saved outcome_yards_scaler.pkl')

# 3. Feature metadata
feature_meta = {
    'feature_cols': FEATURE_COLS,
    'feat_cardinality': FEAT_CARDINALITY,
    'receiver_pos_classes': {0: 'WR', 1: 'TE', 2: 'RB'},
    'bucket_definitions': {
        'feat_play_type':  {'values': {0: 'run', 1: 'pass'}},
        'feat_down':       {'values': {0:'1st',1:'2nd',2:'3rd',3:'4th'}},
        'feat_dist':       {'values': {0:'short(1-2)',1:'medium(3-6)',2:'long(7-10)',3:'very_long(11+)'}},
        'feat_zone':       {'values': {0:'own_deep',1:'own_mid',2:'midfield',3:'opp_mid',4:'red_zone',5:'goal_line'}},
        'feat_score':      {'values': {0:'large_deficit',1:'deficit',2:'close',3:'lead',4:'large_lead'}},
        'feat_qtr':        {'values': {0:'Q1',1:'Q2',2:'Q3',3:'Q4',4:'OT'}},
        'feat_shotgun':    {'values': {0:'under_center',1:'shotgun'}},
        'feat_goal_to_go': {'values': {0:'normal',1:'goal_to_go'}},
        'feat_air_yards':  {'values': {0:'screen/behind',1:'short(0-5)',2:'medium(6-15)',3:'deep(16+)'}},
    }
}
with open(f'{SAVE_DIR}/outcome_feature_meta.json', 'w') as f:
    json.dump(feature_meta, f, indent=2)
print('Saved outcome_feature_meta.json')

# 4. Model config
model_config = {
    'emb_dim': EMB_DIM,
    'hidden': HIDDEN,
    'dropout': DROPOUT,
    'n_features': len(FEATURE_COLS),
    'best_val_loss': best_val_loss,
    'training_seasons': SEASONS,
    'yards_clip': [-10, 50],
}
with open(f'{SAVE_DIR}/outcome_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print('Saved outcome_config.json')

print(f'\nAll artifacts saved to: {SAVE_DIR}')
for fname in sorted(os.listdir(SAVE_DIR)):
    size = os.path.getsize(f'{SAVE_DIR}/{fname}')
    print(f'  {fname:<45} {size/1024:.1f} KB')

## 11: Download weights zip

In [ ]:
from google.colab import files

zip_path = '/tmp/outcome_model_weights_export'
shutil.make_archive(zip_path, 'zip', SAVE_DIR)
files.download(f'{zip_path}.zip')
print('Download started. Extract and place contents in backend/python_backend/outcome_model_weights/')

## Appendix: Quick inference test

In [ ]:
with open(f'{SAVE_DIR}/outcome_yards_scaler.pkl', 'rb') as f:
    scaler_loaded = pickle.load(f)

model_loaded = OutcomeMLP(FEAT_CARDINALITY, EMB_DIM, HIDDEN).to(DEVICE)
model_loaded.load_state_dict(torch.load(f'{SAVE_DIR}/outcome_model.pt', map_location=DEVICE))

print('Cold-load successful.\n')

# Features order: play_type, down, dist, zone, score, qtr, shotgun, goal_to_go, air_yards
SCENARIOS = [
    ('Run, 1st & 10, own 25, tied, Q1',         [0, 0, 2, 1, 2, 0, 0, 0, 0]),
    ('Pass, 3rd & 8, own 30, trailing, Q4',      [1, 2, 2, 1, 1, 3, 1, 0, 2]),  # medium air yards
    ('Pass, 1st & 10, opp 30, tied, Q2 (deep)',  [1, 0, 2, 3, 2, 1, 0, 0, 3]),  # deep shot
    ('Run, 2nd & 1, opp 1, leading (goal line)', [0, 1, 0, 5, 3, 2, 0, 1, 0]),
    ('Pass, 3rd & 15, own 10, large deficit',    [1, 2, 3, 0, 0, 3, 1, 0, 3]),  # desperation deep
]

for label, feats in SCENARIOS:
    x = torch.LongTensor([feats]).to(DEVICE)
    yards, to_prob, td_prob, rp_probs = model_loaded.predict(x, scaler_loaded)
    print(f'{label}')
    print(f'  Predicted yards: {yards[0]:.1f}  |  TO prob: {to_prob[0]*100:.2f}%  |  TD prob: {td_prob[0]*100:.2f}%')
    if feats[0] == 1:  # pass play
        print(f'  Receiver pos:  WR={rp_probs[0][0]*100:.1f}%  TE={rp_probs[0][1]*100:.1f}%  RB={rp_probs[0][2]*100:.1f}%')
    print()